# POC para construir un modelo de predicción de precios de casas

In [1]:
import pandas as pd
import numpy as np
from great_tables import GT
from great_tables import GT, loc, style
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import pickle


## Cargar datos

* Para la prueba de concepto suponemos los datos de la Ciudad de AMES, Iowa.

In [2]:
# Cargar datos
df_train = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")

In [3]:
# Limpiar encabezados de datos
clean_cols = [col.lower() for col in df_train.columns]
df_train.columns = clean_cols

clean_cols = [col.lower() for col in df_test.columns]
df_test.columns = clean_cols

In [31]:
print(df_train.columns)

Index(['id', 'mssubclass', 'mszoning', 'lotfrontage', 'lotarea', 'street',
       'alley', 'lotshape', 'landcontour', 'utilities', 'lotconfig',
       'landslope', 'neighborhood', 'condition1', 'condition2', 'bldgtype',
       'housestyle', 'overallqual', 'overallcond', 'yearbuilt', 'yearremodadd',
       'roofstyle', 'roofmatl', 'exterior1st', 'exterior2nd', 'masvnrtype',
       'masvnrarea', 'exterqual', 'extercond', 'foundation', 'bsmtqual',
       'bsmtcond', 'bsmtexposure', 'bsmtfintype1', 'bsmtfinsf1',
       'bsmtfintype2', 'bsmtfinsf2', 'bsmtunfsf', 'totalbsmtsf', 'heating',
       'heatingqc', 'centralair', 'electrical', '1stflrsf', '2ndflrsf',
       'lowqualfinsf', 'grlivarea', 'bsmtfullbath', 'bsmthalfbath', 'fullbath',
       'halfbath', 'bedroomabvgr', 'kitchenabvgr', 'kitchenqual',
       'totrmsabvgrd', 'functional', 'fireplaces', 'fireplacequ', 'garagetype',
       'garageyrblt', 'garagefinish', 'garagecars', 'garagearea', 'garagequal',
       'garagecond', 'paveddrive

# Modelo Naive por Colonia

* Armamos un modelo naive, que nos sirva de estándar. En este caso suponemos un modelo de precios promedio por colonia.

In [5]:
# Calcular los precios promedio
df_price_preds_by_neighboorhood = (
    df_train
    .groupby(["neighborhood"])
    ["saleprice"]
    .median()
    .to_frame("price")
    .reset_index()
)

In [6]:
# Convertir los datos a parquet.
(df_price_preds_by_neighboorhood
    .to_parquet(
        "../data/analysis/predictions/price_preds_by_neighboorhood.parquet", 
        index=False
    )
)

In [7]:
# Visualizar resultados
df_price_preds_by_neighboorhood

,neighborhood,price
0,Blmngtn,191000.0
1,Blueste,137500.0
2,BrDale,106000.0
3,BrkSide,124300.0
4,ClearCr,200250.0
5,CollgCr,197200.0
6,Crawfor,200624.0
7,Edwards,121750.0
8,Gilbert,181000.0
9,IDOTRR,103000.0


In [9]:
# Construir inferencia para un vecindario
selected_neighborhood = "Blmngtn"
estimated_price = float(df_price_preds_by_neighboorhood.query("neighborhood == @selected_neighborhood")["price"].values[0])

In [32]:
print(estimated_price)

191000.0


# Modelo por Colonia, Cuartos, Baños, Superficie y Estacionamiento

* Construyamos un modelo más complejo, utilizando covariables de interés para el usuario final. 
  - saleprice
  - neighborhood 
  - lotarea
  - bedroomabvgr
  - fullbath 
  - halfbath 
  - garagecars

* Estas son variables que un cliente puede obtener de una casa que le interesa en el mercado.

In [11]:
# Filtar datos
df_m1_lr = (
    df_train
    .filter([
        "saleprice", 
        "neighborhood", 
        "lotarea", 
        "bedroomabvgr",
        "fullbath", 
        "halfbath", 
        "garagecars"
    ])
)

In [12]:
# Construyamos un train test split
X_train, X_test, y_train, y_test = train_test_split(
    df_m1_lr.drop(columns=["saleprice"]), df_m1_lr["saleprice"],
    test_size=0.25, 
    random_state=42
)

In [13]:
# Preprocesamos los datos categóricos
m1_lr_ohe = OneHotEncoder(sparse_output=False, drop="first")
m1_lr_ohe.fit(X_train[["neighborhood"]])
m1_lr_ohe_cols = m1_lr_ohe.get_feature_names_out(["neighborhood"])
df_train_dummies = pd.DataFrame(m1_lr_ohe.transform(X_train[["neighborhood"]]), columns=m1_lr_ohe_cols)
df_test_dummies = pd.DataFrame(m1_lr_ohe.transform(X_test[["neighborhood"]]), columns=m1_lr_ohe_cols)
df_test_dummies

,neighborhood_Blueste,neighborhood_BrDale,neighborhood_BrkSide,neighborhood_ClearCr,neighborhood_CollgCr,neighborhood_Crawfor,neighborhood_Edwards,neighborhood_Gilbert,neighborhood_IDOTRR,neighborhood_MeadowV,...,neighborhood_NoRidge,neighborhood_NridgHt,neighborhood_OldTown,neighborhood_SWISU,neighborhood_Sawyer,neighborhood_SawyerW,neighborhood_Somerst,neighborhood_StoneBr,neighborhood_Timber,neighborhood_Veenker
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
361,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
362,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
363,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# Seleccionamos los datos numéricos
df_train_numbers = X_train.select_dtypes("number")
df_test_numbers = X_test.select_dtypes("number")

In [15]:
# Unimos datos numéricos con categóricos
X_train_clean = pd.concat([df_train_numbers.reset_index(drop=True),df_train_dummies], axis=1)
X_test_clean = pd.concat([df_test_numbers.reset_index(drop=True),df_test_dummies], axis=1)

In [16]:
# Construimos una regresión lineal
m1_lr = LinearRegression()
m1_lr.fit(X_train_clean, y_train)


LinearRegression()

# Guardar artefactos (archivos) del pipeline de inferencia.

* Cada modelo genera artefactos como el modelo entrenado (coeficientes, o puntos de corte) y también las medias, desviaciones estándar de los datos utilizados para entrenar. O en nuestro caso la matriz de diseño del one hot encoder.

* Recuerda que los modelos se entrenan para utilizarse con datos que el modelo no conoce, y tenemos que aplicar a los
  datos de inferencia el mismo proceso para limpiarlos y meterlos al modelo. De otra manera peligras en sobre ajustarlo o tener un data leakage.

In [17]:
# Guardar el modelo entrenado como un pickle file.
with open("../artifacts/m1_lr/model.pkl", "wb") as f:
    pickle.dump(m1_lr, f, protocol=5)

In [18]:
# Guardar el ohe hot encoder ajustado como un pickle file.
with open("../artifacts/m1_lr/ohe.pkl", "wb") as f:
    pickle.dump(m1_lr_ohe, f, protocol=5)

In [19]:
# Probar si podemos cargar el modelo, como un objeto de scikit learn
with open("../artifacts/m1_lr/model.pkl", "rb") as f:
    modelo = pickle.load(f)

In [20]:
# Probar si podemos cargar el one hot encoder, como un objeto de scikit learn
with open("../artifacts/m1_lr/ohe.pkl", "rb") as f:
    ohe = pickle.load(f)

# Real time

* Si queremos utilizar el modelo para hacer inferencias en tiempo de real por una casa, este sería el pipeline de inferencia en tiempo real.

In [33]:
# Construimos un ejemplo artificial como un pandas dataframe
user_entry = pd.DataFrame([{
    'neighborhood': 'IDOTRR',
    'lotarea': 11040,
    'bedroomabvgr': 2,
    'fullbath': 1,
    'halfbath': 1,
    'garagecars': 1
}])

user_entry

,neighborhood,lotarea,bedroomabvgr,fullbath,halfbath,garagecars
0,IDOTRR,11040,2,1,1,1


In [34]:
# Preprocesamos los datos para hacer inferencia
X_user_entry = pd.concat([
    user_entry.select_dtypes("number").reset_index(drop=True),
    pd.DataFrame(m1_lr_ohe.transform(user_entry[["neighborhood"]]), columns=m1_lr_ohe_cols)
    ], 
    axis=1
)
X_user_entry

,lotarea,bedroomabvgr,fullbath,halfbath,garagecars,neighborhood_Blueste,neighborhood_BrDale,neighborhood_BrkSide,neighborhood_ClearCr,neighborhood_CollgCr,...,neighborhood_NoRidge,neighborhood_NridgHt,neighborhood_OldTown,neighborhood_SWISU,neighborhood_Sawyer,neighborhood_SawyerW,neighborhood_Somerst,neighborhood_StoneBr,neighborhood_Timber,neighborhood_Veenker
0,11040,2,1,1,1,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Predecimos
m1_lr.predict(
    X_user_entry
)[0]

# Batch Inference

* Si quieres hacer inferencias en batch, este sería el pipeline de inferencia en batch.

In [25]:
m1_lr_pred = m1_lr.predict(X_test_clean)
(
    GT(X_test.assign(price = m1_lr_pred)).fmt_currency(columns=["price"])
    .tab_header(
        title="House Price Batch Predictions",
        subtitle="2025-01-01 to 2025-01-28"
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns="price"),
    )
)

GT(_tbl_data=     neighborhood  lotarea  bedroomabvgr  fullbath  halfbath  garagecars  \
892        Sawyer     8414             3         1         0           1   
1105      NoRidge    12256             3         2         1           2   
413       OldTown     8960             2         1         0           2   
522       BrkSide     5000             3         2         0           2   
1036       Timber    12898             2         2         0           3   
...           ...      ...           ...       ...       ...         ...   
988        NWAmes    12046             4         2         1           2   
243       SawyerW    10762             3         1         1           1   
1342      CollgCr     9375             3         2         1           2   
1057      NoRidge    29959             3         2         1           2   
1418        NAmes     9204             3         1         1           1   

              price  
892   105342.405043  
1105  301902.775137  
413   128659.834133  
522   173734.490973  
1036  235859.621224  
...             ...  
988   203234.436228  
243   145062.419606  
1342  218602.466485  
1057  320224.482230  
1418  137919.663750  

[365 rows x 7 columns], _body=<great_tables._gt_data.Body object at 0x30011a8d0>, _boxhead=Boxhead([ColInfo(var='neighborhood', type=<ColInfoTypeEnum.default: 1>, column_label='neighborhood', column_align='left', column_width=None), ColInfo(var='lotarea', type=<ColInfoTypeEnum.default: 1>, column_label='lotarea', column_align='right', column_width=None), ColInfo(var='bedroomabvgr', type=<ColInfoTypeEnum.default: 1>, column_label='bedroomabvgr', column_align='right', column_width=None), ColInfo(var='fullbath', type=<ColInfoTypeEnum.default: 1>, column_label='fullbath', column_align='right', column_width=None), ColInfo(var='halfbath', type=<ColInfoTypeEnum.default: 1>, column_label='halfbath', column_align='right', column_width=None), ColInfo(var='garagecars', type=<ColInfoTypeEnum.default: 1>, column_label='garagecars', column_align='right', column_width=None), ColInfo(var='price', type=<ColInfoTypeEnum.default: 1>, column_label='price', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x17d789590>, _spanners=Spanners([]), _heading=Heading(title='House Price Batch Predictions', subtitle='2025-01-01 to 2025-01-28', preheader=None), _stubhead=None, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=0, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=1, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=2, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=3, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=4, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=5, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=6, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=7, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=8, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBody(columns='price', rows=None), grpname=None, colname='price', rownum=9, colnum=None, styles=[CellStyleFill(color='papayawhip')]), StyleInfo(locname=LocBod